In [1]:
from pathlib import Path

import pandas as pd
import yaml


def read_config(config_path: Path = Path("data_config.yaml")) -> dict:
    with config_path.open("r") as stream:
        try:
            return yaml.safe_load(stream)
        except yaml.YAMLError:
            return {}

config = read_config()
config['label_mapping_path'] = "../data/labels_mapping_catboost.csv"


In [2]:
sample_df = pd.read_csv(config.get("sample_path"))


In [3]:
n_participants = len(sample_df)
n_participants


20

In [4]:
sample_df.groupby('Status').describe()


User_id                                                     \
               count       mean       std   min   25%   50%   75%   max   
Status                                                                    
Professional     9.0  14.000000  2.738613  10.0  12.0  14.0  16.0  18.0   
Student         11.0   7.636364  6.360389   1.0   3.5   6.0   8.5  20.0   

             Expirience             ...               note                   \
                  count       mean  ...   75%    max count mean std min 25%   
Status                              ...                                       
Professional        9.0  56.333333  ...  60.0  120.0   0.0  NaN NaN NaN NaN   
Student            11.0  33.272727  ...  36.0   50.0   0.0  NaN NaN NaN NaN   

                          
             50% 75% max  
Status                    
Professional NaN NaN NaN  
Student      NaN NaN NaN  

[2 rows x 24 columns]

In [5]:
sample_df["Notebook use"].value_counts(normalize=True)


Notebook use
Often (more then 3 times a week)    0.8
Sometimes (1-2 times a week)        0.2
Name: proportion, dtype: float64

In [6]:
sum(sample_df.ML) + sum(sample_df.DA)


29

In [7]:
df_hack = pd.read_csv(config.get("dataset_path"), index_col=0)
df_labels = pd.read_csv(config.get("label_mapping_path"), index_col=0)
df_hack = df_hack.merge(df_labels, on='action_id')

df_hack.head()


,action_id,time,session_id,kernel_id,notebook_name,event,cell_index,cell_num,cell_type,cell_source,cell_output,user_id,expert,cell_label
0,0,2023-05-06 10:32:26.282,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,save_notebook,NaN,NaN,NaN,"[\n {\n ""id"": ""35c0b3b694f84140846a21197ea...",NaN,student_7,False,data_preprocessing
1,1,2023-05-06 10:32:55.892,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,helper_functions
2,2,2023-05-06 10:32:56.229,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,finished_execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,"[{""output_type"":""stream"",""size"":23}]",student_7,False,helper_functions
3,3,2023-05-06 10:32:58.048,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,NaN,student_7,False,helper_functions
4,4,2023-05-06 10:32:58.429,709ce80b-90a5-457e-bf6f-b7033a3261b5,bc147b33-fa74-4bff-9f48-c88809c5bdcd,task1.ipynb,finished_execute,35c0b3b694f84140846a21197ea62f68,0.0,code,from mining_extension import check_logging \nc...,"[{""output_type"":""stream"",""size"":23}]",student_7,False,helper_functions


In [8]:
len(df_hack.cell_index.dropna().unique())


2655

In [9]:
len(df_hack)


23622

In [10]:
len(df_hack[df_hack.event == 'execute'])


9207

In [11]:
len(df_hack[df_hack.event == 'delete'])


730

In [12]:
len(df_hack[df_hack.event == 'create'])


1930

In [13]:
len(df_hack[df_hack.event == 'delete'])


730

In [14]:
len(df_hack[~(df_hack.event == 'finished_execute')])


14641

In [15]:
df_hack.loc[df_hack.cell_label.isna(), "cell_label"] = "unlabeled"
ds_steps_distribution_df = (df_hack.cell_label.value_counts(normalize=True) * 100).round(1).to_frame(name="fraction").sort_values("fraction", ascending=False)
ds_steps_distribution_df

,fraction
cell_label,
data_preprocessing,37.1
data_exploration,29.8
comment_only,11.2
modelling,7.3
helper_functions,6.3
load_data,5.5
evaluation,1.7
prediction,1.0
result_visualization,0.1


In [16]:
ds_steps_distribution_df.fraction.sum()

100.1